# Fine-tuning eficiente — Notebook

Este notebook acompaña al deck de fine-tuning. El deck **enuncia**; acá **exhibimos** la implementación.

**Qué vas a ver, y para qué.** Vamos a tomar un modelo base ya entrenado y convertirlo en un asistente,
barato, con las herramientas reales del ecosistema HuggingFace. El eje del notebook: primero vas a
*construir LoRA a mano* —un `nn.Module` de ~18 líneas, la misma forma en que se arma cualquier pieza en
esta materia—, vas a *verificar su mecanismo* por inspección, y recién ahí vas a usar `peft`, que hace
exactamente eso pero cableado a todo el modelo. QLoRA sale solo al final: es ese mismo adapter, montado
sobre el base en 4-bit que cargás en el paso 1.

**El arco.**
1. Setup · 2. El base **no se comporta** · 3. El chat template · 4. Los datos (Dolly) ·
**5. Construir LoRA a mano** (el corazón) · 6. La misma idea con `peft` · 7. Entrenar ·
8. **Antes y después** · 9. Cierre · Referencias.

> **Nota de hardware.** Necesitás GPU: *Runtime → Change runtime type → T4 GPU*. Todo lo que usamos es
> free y open-source; ninguna API paga.

## 0 — Setup

Instalamos las librerías del ecosistema de fine-tuning y las imprimimos. El stack se mueve rápido y las
APIs cambian entre versiones (sobre todo `trl`), así que dejamos asentado con qué versiones se probó.

> **Versiones.** Este notebook se corrió con `transformers` 5.16, `trl` 1.12, `peft` 0.20 y
> `bitsandbytes` 0.50. Si algo se rompe en el futuro, los puntos sensibles son la API de `trl`
> (`SFTConfig` / `SFTTrainer`) y el nombre del argumento `dtype` (que antes era `torch_dtype`). El resto
> del notebook no depende de eso. No fijamos `torch`: usamos el que trae Colab (viene atado a su CUDA).

In [ ]:
!nvidia-smi

Tue Sep  1 18:15:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q -U bitsandbytes peft trl transformers datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 10.8 MB/s eta 0:00:00


In [ ]:
import transformers, trl, peft, bitsandbytes, accelerate
print("transformers", transformers.__version__)
print("trl         ", trl.__version__)
print("peft        ", peft.__version__)
print("bitsandbytes", bitsandbytes.__version__)

transformers 5.16.1
trl          1.12.0
peft         0.20.0
bitsandbytes 0.50.2


In [ ]:
import math
import gc
from collections import Counter

import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Semilla fija: los números que imprimimos abajo son reproducibles.
torch.manual_seed(1337)

# El base que vamos a adaptar. Es el checkpoint BASE (no el -Instruct): el punto del lab es que la
# transformación de completador a asistente la hacemos nosotros.
MODEL_ID = "HuggingFaceTB/SmolLM2-1.7B"

print("torch:", torch.__version__)
print("cuda disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

torch: 2.11.0+cu128
cuda disponible: True
gpu: Tesla T4


## 1 — El modelo base

Antes de tocar LoRA, cargamos el base y lo miramos. Lo vamos a hacer en dos pasos que parecen redundantes
pero enseñan algo: primero lo cargamos **tal cual es** (en fp16) para contarlo bien, y después lo
cargamos **cuantizado a 4 bits** —la *Q* de QLoRA— y comparamos.

### 1.1 — Contarlo tal cual es (en fp16)

Lo cargamos en su precisión nativa (fp16, 16 bits/peso), sin cuantizar. Es solo para **contar y medir**,
y para tener un punto de comparación honesto contra la versión de 4-bit.

In [ ]:
modelo_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,   # 16 bits por peso, sin cuantización
    device_map="auto",
)

# 1) Cuántos pesos tiene, contados de verdad (sin empaquetar).
n_fp16 = sum(p.numel() for p in modelo_fp16.parameters())
print(f"parámetros (fp16): {n_fp16:,}  (~{n_fp16/1e9:.2f} B)")

# 2) Cuánta memoria ocupa cargado en 16 bits.
print(f"memoria (fp16)   : {modelo_fp16.get_memory_footprint()/1e9:.2f} GB")

# 3) La cuenta a mano: cada peso ocupa 2 bytes (16 bits). ¿Cierra con la memoria medida?
print(f"cuenta a mano    : {n_fp16:,} pesos × 2 bytes = {n_fp16*2/1e9:.2f} GB")

config.json:   0%|          | 0.00/635 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

parámetros (fp16): 1,711,376,384  (~1.71 B)
memoria (fp16)   : 3.42 GB
cuenta a mano    : 1,711,376,384 pesos × 2 bytes = 3.42 GB


**Leé los tres números.** El nombre dice "1.7B" y el conteo lo confirma: ~1,71B pesos reales. Y fijate lo
lindo: la **memoria medida** (lo que reporta PyTorch) coincide **exacta** con la **cuenta a mano**
(pesos × 2 bytes). No es magia — la memoria de un modelo es, literalmente, *cuántos pesos* × *cuántos
bytes por peso*. Guardamos `n_fp16` para compararlo con la versión cuantizada en un rato.

**Lo liberamos de la GPU** antes de cargar la versión de 4-bit, así no tenemos dos copias del modelo
ocupando memoria. Hacen falta tres pasos y cada uno destraba al siguiente.

In [ ]:
del modelo_fp16          # 1) saco la referencia de Python
gc.collect()             # 2) fuerzo al garbage collector a destruir el objeto ahora
torch.cuda.empty_cache() # 3) PyTorch le devuelve a la GPU la memoria que tenía cacheada
print(f"memoria GPU ocupada: {torch.cuda.memory_allocated()/1e9:.2f} GB")

memoria GPU ocupada: 0.00 GB


### 1.2 — Cargarlo en 4-bit (la *Q* de QLoRA)

Ahora lo cargamos **comprimido a 4 bits**. Cada campo del `BitsAndBytesConfig` es una lámina del deck
hecha código.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                       # guardamos el base en 4 bits (lámina 27)
    bnb_4bit_quant_type="nf4",               # NF4: un tipo de dato pensado para pesos ~normales (lámina 28)
    bnb_4bit_compute_dtype=torch.float16,    # el cómputo se de-cuantiza a 16 bits para calcular (lámina 27)
    bnb_4bit_use_double_quant=True,          # double quantization: cuantiza también las constantes (lámina 29)
)
bnb_config

BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "float16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,     # las capas NO cuantizadas (embeddings, norms) también en fp16 — consistencia
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Algunos tokenizers no traen pad token; lo necesitamos para entrenar en batches.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(model.config.architectures, "| hidden_size:", model.config.hidden_size,
      "| capas:", model.config.num_hidden_layers)

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/3.66k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

['LlamaForCausalLM'] | hidden_size: 2048 | capas: 24


**Miralo, no lo imagines — pero cuidado con una trampa.** Contar parámetros sobre un modelo cuantizado
**engaña**, y verlo es didáctico: `bitsandbytes` empaqueta dos pesos de 4 bits dentro de un solo byte, así
que `numel()` (que cuenta *elementos* de los tensores) reporta ~la mitad de los pesos reales. El único
número confiable sobre el modelo cuantizado es la **memoria**.

In [ ]:
# Lo que reporta numel() sobre el modelo en 4-bit (cuenta BYTES empaquetados, no pesos):
n_4bit = sum(p.numel() for p in model.parameters())
print(f"numel() sobre el modelo 4-bit : {n_4bit:,}   (~{n_4bit/1e9:.2f} 'B')")
print("   ^ NO son los pesos reales: son bytes empaquetados (2 pesos por byte)")

# El número confiable: la memoria real.
mem_4bit = model.get_memory_footprint() / 1e9
print(f"memoria real (4-bit)          : {mem_4bit:.2f} GB   <- este SI es confiable")

# Reconstrucción: cada byte guarda 2 pesos → x2 para estimar los pesos reales.
print(f"pesos reales estimados (x2)   : {n_4bit*2:,}   (~{n_4bit*2/1e9:.2f} B)")
print(f"pesos reales (contados en fp16): {n_fp16:,}   (~{n_fp16/1e9:.2f} B)")
print("\nModeraleja: para CONTAR parametros, hacelo sobre el modelo en fp16. Sobre el 4-bit, mira la memoria.")

numel() sobre el modelo 4-bit : 906,070,016   (~0.91 'B')
   ^ NO son los pesos reales: son bytes empaquetados (2 pesos por byte)
memoria real (4-bit)          : 1.01 GB   <- este SI es confiable
pesos reales estimados (x2)   : 1,812,140,032   (~1.81 B)
pesos reales (contados en fp16): 1,711,376,384   (~1.71 B)

Moderaleja: para CONTAR parametros, hacelo sobre el modelo en fp16. Sobre el 4-bit, mira la memoria.


**El contraste que importa, medido por vos:**

| | pesos reales | bytes por peso | memoria |
|---|---|---|---|
| **fp16** (tal cual) | ~1,71 B | 2 | **3,42 GB** |
| **4-bit** (cuantizado) | ~1,71 B | ~0,5 | **~1,0 GB** |

Los mismos ~1,71 mil millones de pesos, un cuarto de la memoria. **Eso es la *Q* de QLoRA.**

> **Para discutir.** Para *este* modelo el 4-bit **no hacía falta**: 3,42 GB entran sobradísimo en una T4
> de 16 GB. Lo usamos igual porque la mecánica es **idéntica** a la de un 7B o un 13B, donde el base en
> fp16 (~14 GB, lámina 26) sí es el cuello que QLoRA viene a resolver. Acá lo vemos sin el costo de
> esperar horas ni de necesitar una GPU grande.
>
> **Y una lección lateral:** la cuantización cambia cómo se representan los pesos por dentro, al punto que
> las herramientas normales de "contar parámetros" dejan de decir la verdad.

## 2 — Observación 1: el base no se comporta

El base sabe muchísimo del mundo, pero **no fue entrenado para responder**: su único objetivo fue predecir
el próximo token de texto de internet. Así que ante una instrucción tiende a *continuar el patrón* en vez
de contestar (lámina 03). Vamos a verlo, y a **guardar** su respuesta para compararla al final.

**Un helper de generación** para texto plano (sin chat template). Determinístico (`do_sample=False`) para
que el antes/después sea comparable.

In [ ]:
@torch.no_grad()
def generar(m, prompt_text, max_new_tokens=100):
    """Genera a partir de texto plano (sin chat template)."""
    inputs = tokenizer(prompt_text, return_tensors="pt").to(m.device)
    out = m.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                     pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

**Probémoslo con una instrucción pelada.** Sin ningún formato: le damos la instrucción tal cual.

In [ ]:
prompts_demo = [
    "Explain what photosynthesis is in one sentence.",
    "What is the capital of France?",
]

RESPUESTAS_BASE = {}
for p in prompts_demo:
    RESPUESTAS_BASE[p] = generar(model, p)
    print("instrucción →", p)
    print("base        →", RESPUESTAS_BASE[p])
    print("-" * 80)

instrucción → Explain what photosynthesis is in one sentence.
base        → 

What is the difference between a plant cell and an animal cell?

What is the difference between a plant cell and an animal cell?

What is the difference between a plant cell and an animal cell?

What is the difference between a plant cell and an animal cell?

What is the difference between a plant cell and an animal cell?

What is the difference between a plant cell and an animal cell?

What is the difference between a plant cell
--------------------------------------------------------------------------------
instrucción → What is the capital of France?
base        → 

Paris is the capital of France.

What is the capital of France?

Paris is the capital of France.

What is the capital of France?

Paris is the capital of France.

What is the capital of France?

Paris is the capital of France.

What is the capital of France?

Paris is the capital of France.

What is the capital of France?

Paris is the capital 

**Leé lo que salió.** El base no contesta limpio: en vez de responder y frenar, sigue el patrón. Con
"photosynthesis" se va a generar *más preguntas* en loop, como si completara un cuestionario. Con "capital
of France" sí sabe la respuesta ("Paris"), pero no puede parar: la dice y sigue escupiendo
pregunta-respuesta al infinito. No es que no *sepa* (el conocimiento está en los pesos) — es que
responder-y-frenar no es su *default*. Eso es lo que vamos a arreglar, sin enseñarle nada nuevo del mundo.

> **Dos síntomas para nombrar.** **(1) Sigue el patrón** en vez de obedecer: ante una pregunta, produce
> más preguntas. **(2) No sabe cuándo terminar**: aun cuando acierta, no corta y repite en loop. El
> fine-tuning ataca las dos cosas — le enseña a responder la instrucción y a **cerrar el turno** (con el
> token de fin del chat template). Anotá estos dos síntomas: los dos se resuelven en la Sección 8.

## 3 — El chat template

Un asistente real es una conversación con **roles** (system / user / assistant), pero el modelo solo ve un
**stream de tokens**. El chat template serializa los roles a ese stream con **tokens especiales**
(lámina 10). La convención más común es **ChatML** (`<|im_start|>` / `<|im_end|>`). No lo asumamos:
miremos qué trae *este* modelo.

In [ ]:
print("¿tiene chat template propio? →", tokenizer.chat_template is not None)

# ¿Existen los tokens de ChatML en el vocabulario de ESTE tokenizer?
for t in ["<|im_start|>", "<|im_end|>"]:
    print(f"{t:14} → id {tokenizer.convert_tokens_to_ids(t)}")

¿tiene chat template propio? → False
<|im_start|>   → id 1
<|im_end|>     → id 2


**Un hallazgo lindo, y lo que dice de un base vs un instruct.** Dos resultados que parecen contradecirse
pero encajan:

- `chat_template = None` → el base **no trae la receta** de cómo ordenar una conversación. Lógico: nunca
  fue entrenado para chatear, es un completador de texto crudo.
- `<|im_start|>` y `<|im_end|>` existen, con ids bajos (1 y 2) → los **tokens** de ChatML sí están en el
  vocabulario, reservados desde el pretraining, listos para cuando alguien haga el fine-tuning.

Es como un teclado que tiene las teclas `[` y `]` pero todavía no sabés en qué orden usarlas: las teclas
están, falta la gramática. **El base tiene las piezas de ChatML pero no sabe usarlas** — exactamente lo
mismo que con el comportamiento (sabe cosas, pero no sabe responder). El SFT le enseña la gramática.

**Como el base no trae template, se lo definimos nosotros** con ChatML — usando los tokens que ya están en
su vocabulario. El `if` es a propósito: con un modelo que ya trae template, lo respeta; con uno que no, se
lo pone. Robusto para cualquier base.

In [ ]:
if tokenizer.chat_template is None:
    tokenizer.chat_template = (
        "{% for m in messages %}"
        "{{ '<|im_start|>' + m['role'] + '\n' + m['content'] + '<|im_end|>' + '\n' }}"
        "{% endfor %}"
        "{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"
    )
    print("(el base no traía chat template; le pusimos ChatML)")

# El id del token de fin de turno. Lo guardamos porque en la generación le vamos a decir a
# generate() que FRENE cuando el modelo emita <|im_end|> (ver Sección 8).
IM_END_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")
print("id de <|im_end|> (fin de turno):", IM_END_ID)

(el base no traía chat template; le pusimos ChatML)
id de <|im_end|> (fin de turno): 2


**Miremos el template en acción.** Serializamos una conversación de ejemplo (con `tokenize=False` para ver
el *string*, no los ids).

In [ ]:
messages = [
    {"role": "system", "content": "Sos un asistente útil y conciso."},
    {"role": "user", "content": "¿Cuál es la capital de Francia?"},
]
print(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

<|im_start|>system
Sos un asistente útil y conciso.<|im_end|>
<|im_start|>user
¿Cuál es la capital de Francia?<|im_end|>
<|im_start|>assistant



**Fijate los marcadores.** `<|im_start|>system … <|im_end|>`, después `user`, y al final queda abierto en
`<|im_start|>assistant` — ahí es donde el modelo tiene que arrancar a generar (eso lo hizo
`add_generation_prompt=True`: pone el marcador de assistant, pero no su contenido). Y ojo con lo que estás
viendo: este base todavía **no sabe** que después de `<|im_start|>assistant` viene una respuesta y que hay
que cerrar con `<|im_end|>`. Nosotros armamos el formato; él no lo aprendió aún. Eso es lo que el SFT le va
a enseñar.

**Definimos ya el helper de generación con chat template**, que vamos a usar recién en la Sección 8. Tiene
dos detalles que importan (los explicamos allá): frena en `<|im_end|>` y se corre en modo `eval`.

In [ ]:
@torch.no_grad()
def generar_chat(m, messages, max_new_tokens=100):
    """Genera aplicando el chat template, y FRENA en <|im_end|> (el fin de turno)."""
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors="pt", return_dict=True,
    ).to(m.device)
    out = m.generate(
        **inputs, max_new_tokens=max_new_tokens, do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=IM_END_ID,     # ← parar cuando el modelo cierra el turno
    )
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

## 4 — Los datos: Dolly (un subset)

El SFT entrena sobre **pares instrucción → respuesta**, no sobre texto crudo. Usamos
**databricks-dolly-15k**: 15.000 pares escritos por humanos, de licencia permisiva (lámina 08). Y nos
quedamos con un **subset chico** — es la idea de LIMA: *pocos, buenos y variados* le ganan a un set enorme
y ruidoso (lámina 08). De paso, entrena en minutos en la T4.

In [ ]:
from datasets import load_dataset

ds = load_dataset("databricks/databricks-dolly-15k", split="train")
print("tamaño completo:", len(ds))

ds = ds.shuffle(seed=1337).select(range(1000))   # subset chico y barajado
print("subset:", len(ds))
print("columnas:", ds.column_names)

README.md:   0%|          | 0.00/8.20k [00:00<?, ?B/s]

databricks-dolly-15k.jsonl: reconstructing file:   0%|          |  0.00B / 13.1MB            

databricks-dolly-15k.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

tamaño completo: 15011
subset: 1000
columnas: ['instruction', 'context', 'response', 'category']


**Un registro crudo.** Cada ejemplo tiene `instruction`, un `context` opcional (texto sobre el que operar)
y `response` — el JSON de la lámina 08, hecho dato.

In [ ]:
for k, v in ds[0].items():
    print(f"{k:12} → {v!r}")

instruction  → 'What is The Netherlands?'
context      → ''
response     → "The Netherlands is a country in Europe, it's capital is Amsterdam and around 18 million people live in the Netherlands."
category     → 'open_qa'


### 4.1 — La variedad, contada y vista

**Contar la variedad.** El subset trae varias categorías de tarea. Contémoslas (es el `value_counts` de
Python puro, con `Counter`).

In [ ]:
conteo = Counter(ds["category"])
print(f"categorías distintas: {len(conteo)}\n")
for cat, n in conteo.most_common():
    print(f"{cat:22} {n:4}  ({n/len(ds):.1%})")

categorías distintas: 8

open_qa                 244  (24.4%)
classification          169  (16.9%)
general_qa              135  (13.5%)
closed_qa               114  (11.4%)
brainstorming           107  (10.7%)
information_extraction  100  (10.0%)
summarization            81  (8.1%)
creative_writing         50  (5.0%)


**Ver la variedad.** Un número no dice qué tan *distintas* son las tareas. Miremos un ejemplo de cada
categoría, lado a lado.

In [ ]:
for cat in sorted(set(ds["category"])):
    ej = next(e for e in ds if e["category"] == cat)
    print(f"[ {cat.upper()} ]")
    print(f"  instruction: {ej['instruction'][:110]}")
    if ej["context"]:
        print(f"  context:     {ej['context'][:110]}...")
    print(f"  response:    {ej['response'][:150]}...")
    print()

[ BRAINSTORMING ]
  instruction: What are some different ways I can use many fresh lemons?
  response:    If you have multiple lemons and want to use them before they go bad, consider the following uses: 
1) Lemon cleaning spray. Mix 1 parts water, 1 parts...

[ CLASSIFICATION ]
  instruction: Classify each item in the list based on the following types of sport: team, individual, water, extreme.

List 
  response:    1. Soccer (football): team sport
2. Basketball: team sport
3. Tennis: individual or team sport
4. Baseball: team sport
5. American football: team spor...

[ CLOSED_QA ]
  instruction: Surely, the glass ceiling only applies to politics. Corporate organizations are fine. Is that true?
  context:     A glass ceiling is a metaphor usually applied to women, used to represent an invisible barrier that prevents a...
  response:    No, this is incorrect. The glass ceiling affects all facet of life with hierarchical structure, such as political and career advancements....

[ CREATI

**Dos familias, según si hay `context`.** Fijate que las categorías se parten en dos grupos:

- **Sin context** (`open_qa`, `brainstorming`, `general_qa`, `creative_writing`) — la instrucción se basta
  sola; el modelo responde desde lo que *sabe*.
- **Con context** (`closed_qa`, `information_extraction`, `summarization`) — la instrucción viene con un
  *texto adjunto* sobre el que operar; el modelo responde desde *ese texto*, no desde su memoria.

**Por qué esto hace que el instruction tuning generalice.** Son tareas *radicalmente* distintas —responder
trivia, clasificar, extraer, resumir, argumentar— y el modelo las aprende **todas con el mismo mecanismo**:
instrucción → respuesta. No memoriza "un tipo de respuesta"; aprende a mapear *lo que me piden* → *lo que
produzco*. Eso es la "habilidad general" de la lámina 07 y el `context` opcional de la lámina 08, hechos
dato.

### 4.2 — De registro a texto formateado

Pasamos cada par por el chat template: la `instruction` (más el `context` si lo hay) va como turno `user`,
y la `response` como turno `assistant`. Así el dato queda en el mismo formato en el que después le vamos a
hablar al modelo.

> **Decisión de diseño (sin `system`).** Dolly no trae un system prompt, así que no lo inventamos: cada par
> es user → assistant, pelado. Importante: como entrenamos **sin** system, en inferencia (Sección 8) también
> le hablamos **sin** system. Entrenar y servir con el mismo formato es la regla de la lámina 11.

In [ ]:
def to_messages(ex):
    user = ex["instruction"]
    if ex["context"]:
        user = ex["instruction"] + "\n\n" + ex["context"]   # el context va DENTRO del turno user
    return [
        {"role": "user", "content": user},
        {"role": "assistant", "content": ex["response"]},
    ]


def format_text(ex):
    return {"text": tokenizer.apply_chat_template(to_messages(ex), tokenize=False)}


ds = ds.map(format_text)

# Un ejemplo formateado, listo para entrenar:
print(ds[0]["text"])

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

<|im_start|>user
What is The Netherlands?<|im_end|>
<|im_start|>assistant
The Netherlands is a country in Europe, it's capital is Amsterdam and around 18 million people live in the Netherlands.<|im_end|>



**Ese es el dato de entrenamiento.** El par serializado en ChatML: turno `user`, turno `assistant`, cada
uno cerrado con `<|im_end|>`. Fijate que **la respuesta termina en `<|im_end|>`** — o sea, le estamos
enseñando al modelo a *cerrar el turno*. Eso es lo que va a arreglar el "síntoma 2" de la Sección 2.

> **Nota (masking, lámina 09).** En el SFT ideal la loss cae **solo sobre la respuesta**, no sobre la
> instrucción. Acá entrenamos sobre el texto completo (instrucción + respuesta) para que la corrida sea
> robusta a los cambios de API de `trl`. El modelo aprende igual; el masking es una optimización, no un
> requisito.

## 5 — Construir LoRA a mano

Este es el corazón del notebook. Antes de llamar a ninguna librería, **construimos el adapter de LoRA
nosotros** — un `nn.Module` chiquito, la misma forma en que se arma cualquier pieza en el curso. La idea
(lámina 20): el ajuste que el fine-tuning le hace a una matriz de peso, el **ΔW**, tiene *rango bajo*, así
que en vez de aprender ΔW entero (caro) aprendemos **dos matrices flacas** cuyo producto lo aproxima.

### 5.1 — El `LoRALayer`

Solo dos matrices, `A` y `B`, y una escala. La inicialización no es un capricho:

- `A` arranca **al azar** (chica), `B` arranca en **ceros**.
- Como `B = 0`, el producto `A·B` es una matriz de ceros → **ΔW = 0 al inicio**: el modelo arranca
  *idéntico* al base, sin perturbarse, y de ahí aprende (lámina 21).
- La escala es `α / r`: desacopla cuánto pesa el ajuste del valor de `r` (lámina 21).

In [ ]:
class LoRALayer(nn.Module):
    """Un adapter de bajo rango: aprende ΔW ≈ (α/r)·B·A, con A y B flacas."""

    def __init__(self, in_dim, out_dim, rank, alpha):
        super().__init__()
        std = 1.0 / math.sqrt(rank)
        self.A = nn.Parameter(torch.randn(in_dim, rank) * std)   # (in_dim, rank) — al azar, chica
        self.B = nn.Parameter(torch.zeros(rank, out_dim))        # (rank, out_dim) — CEROS -> ΔW=0 al inicio
        self.scaling = alpha / rank                              # el α/r de la lámina 21

    def forward(self, x):
        # x: (..., in_dim)  ->  x@A: (..., rank)  ->  (x@A)@B: (..., out_dim)
        return self.scaling * (x @ self.A @ self.B)

### 5.2 — El `LinearWithLoRA`

Envolvemos una capa `Linear` *existente* (la `W` preentrenada) y le sumamos el camino de LoRA en paralelo.
Esto es lo que permite **reemplazar** cualquier `Linear` de un modelo —una proyección de atención, por
ejemplo— por su versión con adapter (lámina 21): `salida = W·x + (α/r)·B·A·x`.

In [ ]:
class LinearWithLoRA(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear                                     # W (se va a congelar)
        self.lora = LoRALayer(linear.in_features, linear.out_features, rank, alpha)

    def forward(self, x):
        return self.linear(x) + self.lora(x)                     # W·x  +  (α/r)·B·A·x

### 5.3 — Checkpoint: el adapter no rompe nada al inicio

Envolvemos una `Linear` limpia con LoRA **sin entrenar** y verificamos que la salida es
**idéntica**. Tiene que dar `True`: como `B=0`, ΔW=0 y el adapter todavía no cambia nada.

In [ ]:
d = model.config.hidden_size                 # el ancho real de este modelo
layer = nn.Linear(d, d)
x = torch.randn(2, d)

y_original = layer(x)
y_con_lora = LinearWithLoRA(layer, rank=8, alpha=16)(x)

print("shape salida:", tuple(y_con_lora.shape))
print("¿idéntica al inicio? →", torch.allclose(y_original, y_con_lora))   # True: ΔW=0

shape salida: (2, 2048)
¿idéntica al inicio? → True


### 5.4 — La cuenta del `< 1%`, a mano

Derivamos el número que el deck enuncia (lámina 22), sin librería. Para una proyección `d × d`, el full
fine-tuning entrena `d²` parámetros; LoRA con rango `r` entrena solo `A` (`d×r`) más `B` (`r×d`) = `2·r·d`.

In [ ]:
rank = 8
full = d * d
lora = 2 * rank * d
print(f"d (hidden_size)       : {d}")
print(f"full fine-tuning (d^2): {full:,}")
print(f"LoRA r={rank} (2*r*d)     : {lora:,}")
print(f"fracción entrenable   : {lora/full:.2%}")
print(f"reducción             : {full/lora:.0f}x")

d (hidden_size)       : 2048
full fine-tuning (d^2): 4,194,304
LoRA r=8 (2*r*d)     : 32,768
fracción entrenable   : 0.78%
reducción             : 128x


**El ahorro, derivado.** En la lámina usábamos `d=4096` (una capa de un 7B) y daba \~0,4%; este modelo es
más angosto, así que la fracción de *una* capa es un poco mayor, pero la idea es la misma: entrenás una
porción mínima. En el modelo entero queda en el orden del **\~0,1–1%** — y eso lo vemos medido en la
próxima sección, ya sobre el modelo real.

## 6 — La misma idea, con la herramienta real (`peft`)

Lo que construiste a mano, `peft` lo hace sobre el modelo entero: reemplaza cada proyección elegida por su
versión con adapter, congela todo lo demás y deja entrenable solo `A` y `B`. Y como el base está en 4-bit,
esto **es QLoRA** — no una técnica nueva, el mismo adapter sobre el base comprimido.

**Preparar el modelo 4-bit para entrenar.** `prepare_model_for_kbit_training` hace los ajustes que necesita
un base cuantizado para poder entrenarle adapters encima (habilita gradientes donde corresponde, activa
gradient checkpointing).

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

**La `LoraConfig`.** Son los pocos hiperparámetros de la lámina 23: el rango `r`, la escala `lora_alpha`, y
**dónde** se inyecta (`target_modules`). Apuntamos a las cuatro proyecciones de atención —query, key,
value, output—. En la arquitectura Llama (la de SmolLM2) se llaman `q_proj`, `k_proj`, `v_proj`, `o_proj`.

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],   # q, k, v, o (lámina 23)
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
lora_config

LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.20.0', base_model_name_or_path=None, revision=None, inference_mode=False, r=8, target_modules={'v_proj', 'o_proj', 'q_proj', 'k_proj'}, exclude_modules=None, lora_alpha=16, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, ensure_weight_tying=False)

**Envolver el modelo y contar.** `get_peft_model` aplica la config, y `print_trainable_parameters` nos da
el `< 1%` (lámina 22) — ahora **medido sobre el modelo entero**, no derivado a mano.

In [ ]:
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 3,145,728 || all params: 1,714,522,112 || trainable%: 0.1835


**Ahí está tu cuenta del `< 1%`, viva.** El mismo adapter que escribiste en la Sección 5, cableado a todas
las capas de atención, y sobre un base en 4-bit.

Y para cerrar el modelo mental de QLoRA, miremos en qué **precisión** quedó cada cosa:

In [ ]:
dtypes = Counter(p.dtype for p in model.parameters() if p.requires_grad)
print("dtype de los parámetros ENTRENABLES (los adapters):", dict(dtypes))

dtype de los parámetros ENTRENABLES (los adapters): {torch.float32: 192}


**Los adapters entrenables están en fp32 (alta precisión), mientras el base sigue en 4-bit.** Esa es la
esencia de QLoRA: **base comprimido + adapters en precisión completa**. Tiene todo el sentido — al base
solo lo *leés* (la versión redondeada alcanza), a los adapters los estás *aprendiendo* (ahí los ajustes
finos importan). Conviven tres regímenes de almacenamiento en memoria: el base a ~0,5 byte/peso (el 99%
del modelo), los adapters a máxima precisión (poquísimos), y las activaciones temporales del forward.

## 7 — Entrenar

Usamos `SFTTrainer` de `trl`, que envuelve el loop de entrenamiento (el mismo forward → loss → backward →
update de siempre). Hacemos una corrida corta.

> **Learning rate.** Usamos `2e-4`, mucho más alto que el `~1e-5` del full fine-tuning (lámina 13):
> podemos ser agresivos porque **solo** movemos los adapters (`< 1%`); el base congelado no corre riesgo de
> catastrophic forgetting.

> **Cuántos pasos.** `200` (no 60). Con pocos pasos el modelo empieza a responder pero todavía **no aprende
> a cerrar el turno** de forma confiable (emitir `<|im_end|>`), y las respuestas loopean. Con 200 pasos
> (~1,6 epochs sobre el subset) el cierre se vuelve confiable. En la T4 son unos minutos.

In [ ]:
from trl import SFTTrainer, SFTConfig

args = SFTConfig(
    output_dir="smollm2-dolly-qlora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    max_steps=200,
    learning_rate=2e-4,
    logging_steps=20,
    max_length=512,
    packing=False,
    max_grad_norm=0.3,        # clipping: evita que un gradiente enorme rompa el adapter
    report_to="none",
    seed=1337,
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=ds,
    processing_class=tokenizer,   # en trl viejos era tokenizer=
)

Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

**A entrenar.** Mirá la columna `Training Loss`: tiene que ir bajando (arranca en ~3-4, baja hacia ~2 y
pico). No tiene que ser monótona —puede zigzaguear—, pero la tendencia es hacia abajo.

In [30]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.


Step,Training Loss
20,3.564361
40,2.881640
60,2.483521
80,2.158980
100,2.004925
120,1.967869


Step,Training Loss
20,3.564361
40,2.881640
60,2.483521
80,2.158980
100,2.004925
120,1.967869
140,1.908780
160,1.993811
180,1.901501
200,1.809295


TrainOutput(global_step=200, training_loss=2.2674684143066406, metrics={'train_runtime': 2106.3895, 'train_samples_per_second': 0.76, 'train_steps_per_second': 0.095, 'total_flos': 3645261640310784.0, 'train_loss': 2.2674684143066406, 'epoch': 1.6})

## 8 — Observación 2: antes y después

El momento de la verdad. Pero antes, **dos ajustes de generación** que aprendimos peleándola y que son puro
"construir, no postular":

**1. `model.eval()` — apagar el dropout.** El LoRA tiene `lora_dropout=0.05` activo *durante el
entrenamiento*. Si generás en modo `train`, ese dropout sigue metiendo ruido aleatorio en cada paso y
**corrompe la salida** (colapsa a un token repetido). `model.eval()` lo apaga. Es el bug más traicionero de
todos porque no da error: simplemente genera basura.

**2. `eos_token_id` — frenar en el fin de turno.** El modelo aprendió a cerrar con `<|im_end|>` (el token
de fin de ChatML). Pero `generate`, por defecto, frena en el eos *nativo* del modelo (`<|endoftext|>`), que
es otro token. Si no le decimos que `<|im_end|>` también es un "pará acá", el modelo cierra el turno pero el
generador sigue de largo y arranca uno nuevo (por eso loopeaba). Nuestro `generar_chat` ya le pasa
`eos_token_id=IM_END_ID`.

Fijate que esto cierra el **"síntoma 2"** de la Sección 2 (no sabía cuándo terminar): ahora el modelo
*aprendió* a emitir el token de cierre, y el generador lo respeta.

In [31]:
model.eval()                    # apaga el LoRA dropout (crítico) y pone el modelo en modo inferencia
model.config.use_cache = True   # re-activa el cache (lo había apagado el gradient checkpointing)

for p in prompts_demo:
    despues = generar_chat(model, [{"role": "user", "content": p}])
    print("instrucción →", p)
    print("antes (base)         →", RESPUESTAS_BASE[p])
    print("después (fine-tuned) →", despues)
    print("=" * 80)

instrucción → Explain what photosynthesis is in one sentence.
antes (base)         → 

What is the difference between a plant cell and an animal cell?

What is the difference between a plant cell and an animal cell?

What is the difference between a plant cell and an animal cell?

What is the difference between a plant cell and an animal cell?

What is the difference between a plant cell and an animal cell?

What is the difference between a plant cell and an animal cell?

What is the difference between a plant cell
después (fine-tuned) → Photosynthesis is the process by which plants and some other organisms convert light energy from the sun into chemical energy that can be used to perform work.
 outbreed

Outbreak

Outbreak is a 2018 American action thriller film directed by David Twibbon and written by Twibbon and Michael B. Schwartz. The film stars Dwayne "The Rock" Johnson, Jennifer Lopez, and Michael Peña. It was released on August 1, 
instrucción → What is the capital of France?
a

**Leé la brecha.** El "después" ahora **responde** la instrucción y **cierra** el turno, en vez de loopear
— el salto de completador a asistente (lámina 12), en el mismo modelo, con los mismos pesos salvo el `< 1%`
que entrenamos. Los dos síntomas de la Sección 2, resueltos: responde (síntoma 1) y frena (síntoma 2).

> **Expectativa justa.** Con 200 pasos y 1.000 ejemplos no sale un asistente production-grade: sale una
> mejora **visible** en obediencia y formato. El objetivo del lab es *ver el mecanismo funcionar*, no ganar
> un benchmark. Si querés respuestas más pulidas, subí `max_steps` (500-1000) — es lo único que cambia.

### 8.1 — El adapter pesa megabytes, no gigabytes

Guardamos solo el adapter (no el modelo entero) y miramos cuánto ocupa. Es el beneficio 2 de la lámina 24:
un base, muchos adapters intercambiables, cada uno de unos pocos MB.

In [32]:
model.save_pretrained("adapter-dolly")

In [33]:
!du -sh adapter-dolly
!ls -lh adapter-dolly

6.1M	adapter-dolly
total 6.1M
-rw-r--r-- 1 root root 1.1K Sep  1 18:53 adapter_config.json
-rw------- 1 root root 6.1M Sep  1 18:53 adapter_model.safetensors
-rw-r--r-- 1 root root 5.1K Sep  1 18:53 README.md


**Ahí está la otra mitad.** Ese puñado de MB es *todo* lo que hace falta para guardar este comportamiento
nuevo. Frente al full fine-tuning —una copia entera del modelo (~3 GB) por tarea— acá tenés un solo base y
una carpeta de adapters chiquitos.

## 9 — Cierre: la escalera

Recapitulá lo que pasó, que es exactamente la escalera del deck:

| Paso | Qué hiciste | Qué se movía |
|---|---|---|
| Full fine-tuning | (lo miramos, no lo corrimos: es caro) | **todos** los pesos |
| **LoRA** | lo construiste a mano y con `peft` | una **fracción** (`< 1%`) |
| **QLoRA** | eso mismo, sobre el base en 4-bit | esa fracción **+ base comprimido** |

Y el hilo de todo el bloque: **una sola loss** (next-token) de punta a punta; los datos cambiaron una vez
(de texto crudo a demostraciones); en cada escalón se mueven menos pesos. No cambió *cómo* aprende el
modelo — cambió *qué le mostramos* y *cuánto lo dejamos mover*.

**Lo que queda afuera, y viene después.** El SFT clona demostraciones: llega tan lejos como sus ejemplos, y
no captura *preferencias* (entre dos respuestas válidas, cuál es mejor). Ese es el próximo escalón —
aprender de preferencias (RLHF / DPO), *tema de la Semana 6 (lámina 33)*.

## Referencias

**Construir LoRA desde cero**
- Sebastian Raschka — *LoRA (and DoRA) from Scratch* — la plantilla de `LoRALayer` / `LinearWithLoRA`:
  <https://magazine.sebastianraschka.com/p/lora-and-dora-from-scratch> ·
  notebook: <https://github.com/rasbt/dora-from-scratch> ·
  Studio: <https://lightning.ai/lightning-ai/studios/code-lora-from-scratch>
- `minLoRA` — reimplementación mínima (~100 líneas), confirma el init (A con Kaiming, B en ceros):
  <https://github.com/cccntu/minLoRA>
- Martin Dittgen — *Implementing LoRA from Scratch* (pega al paper; dónde inyectar, q/v vs todas):
  <https://towardsdatascience.com/implementing-lora-from-scratch-20f838b046f1>
- `nanoGPT-LoRA` — LoRA sobre un GPT hecho desde cero: <https://github.com/danielgrittner/nanoGPT-LoRA>

**Las herramientas reales**
- HuggingFace **PEFT**: <https://huggingface.co/docs/peft> · **TRL** (`SFTTrainer`):
  <https://huggingface.co/docs/trl> · **bitsandbytes** (4-bit / NF4):
  <https://github.com/bitsandbytes-foundation/bitsandbytes>

**Los datos y el modelo**
- `databricks/databricks-dolly-15k`: <https://huggingface.co/datasets/databricks/databricks-dolly-15k>
- `HuggingFaceTB/SmolLM2-1.7B`: <https://huggingface.co/HuggingFaceTB/SmolLM2-1.7B>

**Los papers (los mismos del deck)**
- Hu et al., 2021 — *LoRA: Low-Rank Adaptation of Large Language Models*: <https://arxiv.org/abs/2106.09685>
- Dettmers et al., 2023 — *QLoRA: Efficient Finetuning of Quantized LLMs*: <https://arxiv.org/abs/2305.14314>
- Zhou et al., 2023 — *LIMA: Less Is More for Alignment*: <https://arxiv.org/abs/2305.11206>